In [0]:
def bc_ev_harmonized():
    from pyspark.sql import functions as F
    df_bc_silver = (spark.readStream
                    .table("ev_spark.silver.bc_ev")
    )

    df_postal_codes = spark.table("ev_spark.silver.postal_codes")

    df_bc_harmonized = (df_bc_silver
                            .join(
                                df_postal_codes,
                                F.lower(df_bc_silver.city) == F.lower(df_postal_codes.City),
                                "inner"
                            )
                            .select(df_bc_silver.province, "fsa", "ev_count")
    )
    return df_bc_harmonized

In [0]:
def evCombined():
    df_bc_harmonized = bc_ev_harmonized()
    df_on_silver = spark.readStream.table("ev_spark.silver.ontario_ev")
    df_ev_combined = df_on_silver.unionByName(df_bc_harmonized)
    return df_ev_combined


In [0]:
def writeEvCombinedToSilver(df):
    (df
        .writeStream
        .format("delta")
        .option("checkpointLocation", "/Volumes/ev_spark/myvol/checkpoint/chkpt/ev_combined_silver/")
        .trigger(availableNow=True)
        .toTable("ev_spark.silver.ev_combined")
    )

In [0]:
df_ev_combined = evCombined()
writeEvCombinedToSilver(df_ev_combined)